# Download ERA5 offshore waves at the goals

ERA5 ocean waves (ECMWF; hourly, 0.5°, 1940 onwards) from the Earth Data Hub
zarr, read only at the goal nodes and written in the HyWaves names on
`(time, goal)`, like any other offshore source:

- bulk `hs`, `tp`, `dir`, `spr`;
- partitions `phs{i}`, `ptp{i}`, `pdir{i}`, `pspr{i}`: 0 = wind sea, 1–3 = the
  three swell partitions.

ERA5 has no partition peak periods: it gives the partition **mean** periods
(Tm-1,0). *Partition peak periods* below converts them with the Tp / Tm-1,0
ratio measured on ERA5 itself, so the zarr holds peak periods like any other
source. The three swells share the total-swell spread.

**Key:** set the `EDH_API_KEY` environment variable (e.g. in `~/.bashrc`), or
type it when asked below. Never write it in the notebook.

**Prerequisites:** `04_Select_Goals_by_Hindcasts.ipynb` (the goals JSON, with
an `era5` node per goal) and an `ERA5` entry in `offshore_waves`.
**Next:** `07b_Satellite_Calibration.ipynb` with `offshore_name = "ERA5"`.

In [ ]:
import getpass
import os

import matplotlib.pyplot as plt
import xarray as xr
from bluemath_tk.waves.hywaves.offshore import (
    EDH_KEY_VARIABLE,
    era5_at_goals,
    jonswap_tm10_ratio,
    open_era5_waves,
    partition_peak_periods,
    peak_period_samples,
)
from bluemath_tk.waves.hywaves.plots import plot_peak_period_ratio
from utils.config import get_offshore_wave, load_goals, resolve

## Configuration

In [ ]:
offshore_name = "ERA5"  # offshore_waves entry this zarr feeds
period = slice("1979", "2023")  # time window (ERA5 starts in 1940)

out_path = resolve(get_offshore_wave(offshore_name)["data_path"])
if not os.environ.get(EDH_KEY_VARIABLE):
    os.environ[EDH_KEY_VARIABLE] = getpass.getpass("Earth Data Hub key: ")
print(f"output : {out_path}")
print(f"period : {period.start} – {period.stop}")

## Goal nodes

The ERA5 node of each goal from the goals JSON (`hindcast.era5`, chosen in 04);
the nearest ERA5 grid point is read.

In [ ]:
goals = load_goals()
nodes = {gid: goal["hindcast"][offshore_name.casefold()] for gid, goal in goals.items()}
goal_ids = [int(gid) for gid in nodes]
lon = [node["longitude"] for node in nodes.values()]
lat = [node["latitude"] for node in nodes.values()]
list(zip(goal_ids, lon, lat))

## Download

All variables and partitions at the goal nodes (about a minute per decade).

In [ ]:
era5 = era5_at_goals(open_era5_waves(), lon, lat, goal_ids, time=period)
era5

## Partition peak periods

The metamodel is forced with peak periods; ERA5 gives the partitions' mean
period Tm-1,0. For a JONSWAP spectrum Tp / Tm-1,0 is 1.17 (γ = 1,
Pierson-Moskowitz), 1.11 (γ = 3.3) and 1.07 (γ = 7, a narrow swell).

Measured on ERA5 itself: at the hours one partition holds at least 90 % of the
energy, the bulk peak period (`pp1d`) is that partition's peak, so
`tp / ptp{i}` is its ratio. The medians (wind sea, and the swells pooled)
convert `ptp{i}`.

In [ ]:
samples = peak_period_samples(era5, min_fraction=0.9)
ratios = samples.groupby("partition")["ratio"].median().to_dict()
theory = {f"JONSWAP γ={g:g}": 1 / jonswap_tm10_ratio(g) for g in (1.0, 3.3, 7.0)}
fig = plot_peak_period_ratio(samples, theory, title="ERA5 · Tp / Tm-1,0")
plt.show()
ratios

In [ ]:
era5_tp = partition_peak_periods(era5, ratios)
era5_tp["ptp1"].attrs

In [ ]:
out_path.parent.mkdir(parents=True, exist_ok=True)
era5_tp.chunk({"time": -1, "goal": 1}).to_zarr(out_path, mode="w")
print(f"written {out_path}")

## Check

In [ ]:
check = xr.open_zarr(out_path)
summary = check[["hs", "phs0", "phs1", "phs2", "phs3"]].mean("time").to_dataframe()
summary["missing_hs"] = check["hs"].isnull().mean("time").to_series()
summary.round(3)